# Super CONUS NGWPC Hydrofabric Demo
This notebook walks through the following rejected PI-8 acceptance criteria for the Super CONUS.

- ensuring rivers can be represented as a directed acyclic graphs for routing
- connectivity checks
- POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses.
- Every attempt will be made to maximize The NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

## Ensuring rivers can be represented as a directed acyclic graphs for routing / connectivity checks

The following code builds a graph from NHF 1.1.3 and tests with the networkx `is_directed_acyclic_graph` function. 

In [ ]:
"""Check that the NHF 1.1.3 nexus-mediated graph is a DAG."""

import sqlite3

import networkx as nx

GPKG = "../data/nhf_1.1.3.gpkg"

con = sqlite3.connect(GPKG)

total_flowpaths = con.execute("SELECT count(*) FROM flowpaths").fetchone()[0]
total_nexuses = con.execute("SELECT count(*) FROM nexus").fetchone()[0]

fp_to_nex = con.execute("SELECT fp_id, dn_nex_id FROM flowpaths").fetchall()
nex_to_fp = con.execute("SELECT nex_id, dn_fp_id FROM nexus WHERE dn_fp_id IS NOT NULL").fetchall()

con.close()

G = nx.DiGraph()

fp_ids = set()
nex_ids = set()

# IMPORTANT: The fp_to_nex edge construction adds all outlet flowpaths because all flowpaths (including outlet flowpaths) have a dn_nex_id
for fp, nex in fp_to_nex:
    fp_node = f"fp_{fp}"
    nex_node = f"nex_{nex}"
    G.add_edge(fp_node, nex_node)
    fp_ids.add(fp_node)
    nex_ids.add(nex_node)

# IMPORTANT: The 12,035 outlet nexuses that are excluded from nex_to_fp are included in fp_to_nex and so they still get included in the graph in the loop above.
for nex, fp in nex_to_fp:
    nex_node = f"nex_{nex}"
    fp_node = f"fp_{fp}"
    G.add_edge(nex_node, fp_node)
    nex_ids.add(nex_node)
    fp_ids.add(fp_node)

print(f"Total flowpaths in NHF:    {total_flowpaths:,}")
print(f"Total nexuses in NHF:      {total_nexuses:,}")
print(f"Flowpath nodes in graph:   {len(fp_ids):,}")
print(f"Nexus nodes in graph:      {len(nex_ids):,}")
print(f"Total graph nodes:         {G.number_of_nodes():,}")
print(f"Total graph edges:         {G.number_of_edges():,}")
print()

is_dag = nx.is_directed_acyclic_graph(G)
print(f"Is DAG: {is_dag}")

if not is_dag:
    cycles = list(nx.simple_cycles(G))
    print(f"Found {len(cycles):,} cycle(s)")
    for c in cycles[:5]:
        print(f"  {c}")


In [ ]:
"""Visualize NHF 1.1.3 nexus-mediated graph structure for a subset of flowpaths."""

import sqlite3
from collections import defaultdict, deque

import matplotlib.pyplot as plt
import networkx as nx

GPKG = "../data/nhf_1.1.3.gpkg"
OUTLET_FP_ID = 641
MAX_NODES = 120

con = sqlite3.connect(GPKG)

fp_to_nex_all = con.execute("SELECT fp_id, dn_nex_id FROM flowpaths").fetchall()
nex_to_fp_all = con.execute(
    "SELECT nex_id, dn_fp_id FROM nexus WHERE dn_fp_id IS NOT NULL"
).fetchall()

con.close()

fp_to_fp_all = []
nex_dn = {nex: fp for nex, fp in nex_to_fp_all}
for fp, nex in fp_to_nex_all:
    dn_fp = nex_dn.get(nex)
    if dn_fp is not None:
        fp_to_fp_all.append((fp, dn_fp))

upstream = defaultdict(list)
for fp, to_fp in fp_to_fp_all:
    upstream[int(to_fp)].append(int(fp))

fp_subset = set()
queue = deque([OUTLET_FP_ID])
while queue and len(fp_subset) < MAX_NODES:
    node = queue.popleft()
    if node not in fp_subset:
        fp_subset.add(node)
        queue.extend(upstream.get(node, []))

fp_nex_map = {fp: nex for fp, nex in fp_to_nex_all if fp in fp_subset}
nex_subset = set(fp_nex_map.values())

fp_to_nex_sub = [(fp, nex) for fp, nex in fp_to_nex_all if fp in fp_subset]
nex_to_fp_sub = [(nex, fp) for nex, fp in nex_to_fp_all if nex in nex_subset and fp in fp_subset]

print(f"{len(fp_subset)} flowpaths, {len(nex_subset)} nexus nodes")

FP_COLOR = "#4A90D9"
NEX_COLOR = "#E8783A"

G = nx.DiGraph()
for fp in fp_subset:
    G.add_node(f"fp_{fp}", ntype="fp")
for nex in nex_subset:
    G.add_node(f"nex_{nex}", ntype="nex")
fp_nex_edges = [(f"fp_{fp}", f"nex_{nex}") for fp, nex in fp_to_nex_sub]
nex_fp_edges = [(f"nex_{nex}", f"fp_{fp}") for nex, fp in nex_to_fp_sub]
G.add_edges_from(fp_nex_edges)
G.add_edges_from(nex_fp_edges)

try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except Exception:
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

fp_nodes = [n for n in G if G.nodes[n].get("ntype") == "fp"]
nex_nodes = [n for n in G if G.nodes[n].get("ntype") == "nex"]

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_title(
    f"NHF 1.1.3 nexus-mediated graph ({len(fp_subset)} flowpaths near fp_id={OUTLET_FP_ID})",
    fontsize=12, fontweight="bold",
)

nx.draw_networkx_nodes(G, pos, nodelist=fp_nodes, node_color=FP_COLOR, node_size=40, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=nex_nodes, node_color=NEX_COLOR, node_size=25, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=fp_nex_edges, ax=ax, edge_color=FP_COLOR,
                       arrows=True, arrowsize=6, width=1.0, alpha=0.5)
nx.draw_networkx_edges(G, pos, edgelist=nex_fp_edges, ax=ax, edge_color="#D62728",
                       arrows=True, arrowsize=10, width=2.0, style="dashed",
                       connectionstyle="arc3,rad=0.3")

ax.legend(handles=[
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=FP_COLOR, markersize=8, label=f"flowpath ({len(fp_nodes)})"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=NEX_COLOR, markersize=8, label=f"nexus ({len(nex_nodes)})"),
    plt.Line2D([0], [0], color=FP_COLOR, linewidth=1, label="fp → nex"),
    plt.Line2D([0], [0], color="#D62728", linewidth=1.5, linestyle="dashed", label="nex → fp"),
], loc="upper left", fontsize=8)
ax.axis("off")

fig.tight_layout()
fig.savefig("output_nhf113_graph.png", dpi=150, bbox_inches="tight")


## POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses.

### Waterbodies / Lakes
The `lakes` layer was built in NHF to be a 1:1 representation of NWM operational waterbodies. The `lakes` layer retains all data from Hydrofabric 2.2 and the `nwm_lakes.gpkg` operational lakes polygon layer provided by OWP in January 2026. The `nwm_lakes.gpkg` includes CONUS, Puerto Rico, and Hawaii lakes. This gpkg was clipped to CONUS borders and saved for an NHF input as `nwm_lakes_sconus_input.gpkg`.

While the lake point geometry may not lie directly in a lake polygon, the matching COMID polygon geometry is used to select the most downstream flowpath intersection. The most downstream flowpath is identified using the minimum hydrosequence. The downstream nexus of this flowpath becomes the `dn_nex_id` associated with each lake.

In [ ]:
import geopandas as gpd

path_nhf = "../data/nhf_1.1.3.gpkg"
path_lakes = "../data/nwm_lakes_sconus_input.gpkg"

def compare_lakes(nhf_path: str, nwm_path: str, domain: str, id_field: str):
    """Compare if lakes are present in an NWM source file and an NHF gages layer"""
    gdf_nwm = gpd.read_file(nwm_path)
    gdf_nhf = gpd.read_file(nhf_path, layer="lakes")
    print(f"{domain} NHF lakes: {len(gdf_nhf)}")
    print(f"{domain} NWM lakes: {len(gdf_nwm)}")
    print(f"{domain} lakes COMID in NWM: {len(gdf_nhf.loc[gdf_nhf['lake_id'].isin(gdf_nwm[id_field])])}")
    display(gdf_nhf.head())

compare_lakes(path_nhf, path_lakes, "CONUS", "newID" )

### Gages
Gages were extracted from routelink and USGS. Routelink files were downloaded from NWM v3.0.18, converted to GPKG in EPSG:4326, and extracted rows with populated gage ID field. 

If upstream area information was available, gages were matched to flowpaths/divides using it. If it was not available, gages were matched to nearest flowpath. See connectivity columns in tables below (fp_id, virtual_fp_id, dn_nex_id, dn_virtual_nex_id).

Canadian gages were missing in the original NHF delivery. Including all routelink gages brought all necessary Canadian.

In [ ]:

path_nhf = "../data/nhf_1.1.3.gpkg"
path_routelink = "../data/RouteLink_CONUS_EPSG4326.gpkg"

def compare_gages(nhf_path: str, routelink_path: str, domain: str):
    """Compare if gages are present in routelink and an NHF gages layer"""
    gdf_routelink = gpd.read_file(routelink_path)

    # extract rows with gages
    gdf_routelink["gages"] = gdf_routelink["gages"].str.strip()
    gdf_routelink = gdf_routelink.loc[gdf_routelink["gages"] != ""].copy()

    gdf_gages = gpd.read_file(nhf_path, layer="gages")
    print(f"{domain} NHF gages: {len(gdf_gages)}")
    print(f"{domain} Routelink gages: {len(gdf_routelink)}")
    print(f"{domain} gage ID in Routelink: {len(gdf_gages.loc[gdf_gages['site_no'].isin(gdf_routelink['gages'])])}")
    display(gdf_gages.head())

compare_gages(path_nhf, path_routelink, "Super CONUS")

The following function was used to extract gages from Routelink.

In [ ]:
# Function used in NHF-builds to extract routelink gages - this is for demonstration purposes only
from pathlib import Path

import pandas as pd


def append_from_routelink(
    gdf: gpd.GeoDataFrame, routelink: Path, id_col_name: str, shape: Path | None
) -> gpd.GeoDataFrame:
    """Append gages from RouteLink file to GeoDataFrame

    Use ogr2ogr to convert NC file to GPKG and add EPSG:4326 georef i.e. ogr2ogr RouteLink.gpkg RouteLink.nc -t_srs EPSG:4326 -s_srs EPSG:4326

    Parameters
    ----------
    gdf: GeoDataFrame
        Input dataframe to append to
    routelink : Path
        RouteLink file to extract from
    id_col_name: str
        Column to pull from for site_no in RouteLink
    shape: Path | None
        Shapefile to use for clipping
    """
    gages = gpd.read_file(routelink).to_crs(gdf.crs)

    # first get gages only
    gages = gages.loc[gages[id_col_name].str.strip() != ""].copy()

    # then check intersection if requested
    if shape:
        # Get boundary to clip to
        shp = gpd.read_file(shape).to_crs(gdf.crs)
        merged_geom = shp["geometry"].union_all()
        gages = gages.loc[gages["geometry"].intersects(merged_geom), :].copy()

    gages = gages.rename(columns={id_col_name: "site_no"})
    gages["site_no"] = gages["site_no"].str.strip()

    gages = gpd.GeoDataFrame(gages[["geometry", "site_no"]][~gages["site_no"].isin(gdf["site_no"])].copy())
    # logger.info(f"gages: added {len(gages)} gages from RouteLink not already present in dataset") # commented for missing imports in demonstration
    gages["status"] = "routelink"
    gages = pd.concat([gdf, gages])
    gages["geometry"] = gages["geometry"].force_2d()

    return gages

## Gage / Lake Flowpath Association
In the following example, a lake COMID and gage are plotted with their associated downstream nexus. 

Below, a map will display a series of NHF information in Puerto Rico.

- A lake point is plotted in red. The point geometry is in the centroid of the lake polygon. This centroid is not in the lake geometry.
- A lake polygon (COMID 800043415) is plotted in bright blue. It follows the outline of the basemap lake.
- A gage point (USGS site no 50027200) is plotted in green to the west of a dam.
- All NHF flowpaths are plotted in navy blue.
- All NHF nexus are plotted in mageneta.
- The downstream nexus that gage and lakes are mapped to circles a magenta nexus in orange.

Although the lake point does not intersect the polygon, it is mapped to the most downstream flowpath intersecting the polygon. 

The gage is mapped to the nearest flowpath. If upstream gage area is available, gages can be mapped to the most similar upstream area nexus.

In this case, both lake and gage outlets are correctly mapped to the outlet of the lake.

Because the lakes are mapped to nexus using the lake polygon, identifying lake point nexuses may show associated nexus far from the point geometry itself.

In [ ]:
import geopandas as gpd

path_nex_demo = "../data/lake_gage_nexus.gpkg"

gdf_lk = gpd.read_file(path_nex_demo, layer="lakes")
gdf_fp = gpd.read_file(path_nex_demo, layer="flowpaths")
gdf_nex = gpd.read_file(path_nex_demo, layer="nexus")
gdf_gage = gpd.read_file(path_nex_demo, layer="gages")
gdf_lk_poly = gpd.read_file(path_nex_demo, layer="nwm_lakes")

In [ ]:
comid = 166766861
gage = "04232482"

lk_pt = gdf_lk.loc[gdf_lk["lake_id"] == comid, :].copy()
lk_poly = gdf_lk_poly.loc[gdf_lk_poly["newID"] == comid, :].copy()
gage_pt = gdf_gage.loc[gdf_gage["site_no"] == gage, :].copy()

nex_id_lk = lk_pt["dn_nex_id"].values
nex_id_gage = gage_pt["dn_nex_id"].values

print(f"Nexus ID for lake {comid}: {nex_id_lk}")
print(f"Nexus ID for gage {gage}: {nex_id_gage}")

nex_lk = gdf_nex.loc[gdf_nex["nex_id"].isin(nex_id_lk), :].copy()
nex_gage = gdf_nex.loc[gdf_nex["nex_id"].isin(nex_id_gage), :].copy()

m = lk_poly.explore(color="#20b9d0", name="Lake Polygon")
m = lk_pt.explore(m=m, color="red", name="Lake Point", marker_kwds={"radius":10})
m = gdf_fp.explore(m=m, color="#290398", name="Flowpaths")
m = gdf_nex.explore(m=m, color="#d309c3", marker_kwds={"radius": 5}, name="Nexus")
m = nex_lk.explore(m=m, color="orange",marker_kwds={"radius":10}, name="Lake Nexus")
m = nex_gage.explore(m=m, color="black",marker_kwds={"radius":10}, name="Lake Nexus")
m = gage_pt.explore(m=m, color="green", marker_kwds={"radius":5}, name="Gage", legend=True)
m

## Every attempt will be made to maximize The NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

The following code plots the distribution of flowpaths lengths in NHF 1.1.3 compared to the link segment lenghts generated for t-route. While the NHF geometry itself can deviate from the 250-350 meter discretization range, the links_nodes.gpkg used for t-route consistently complies.

In [ ]:
"""Plot flowpath length distribution (nhf_1.1.3.gpkg) vs link segment lengths (links_nodes.gpkg)."""

import sqlite3

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

conn = sqlite3.connect("../data/nhf_1.1.3.gpkg")
nhf_km = np.array([r[0] for r in conn.execute("SELECT length_km FROM flowpaths WHERE length_km > 0")])
conn.close()

links = gpd.read_file("../data/links_nodes.gpkg", layer="links")
link_km = (links.length / 1000).values
link_km = link_km[link_km > 0]
del links

datasets = [
    (nhf_km, "NHF v1.1.3 flowpaths"),
    (link_km, "Link segments"),
]
COLORS = ["steelblue", "darkorange"]

bins = np.logspace(-3, np.log10(max(d[0].max() for d in datasets)), 100)
fig, ax = plt.subplots(figsize=(10, 6))

for i, (lengths, name) in enumerate(datasets):
    clipped = lengths[lengths >= 1e-3]
    ax.hist(clipped, bins=bins, weights=np.ones(len(clipped)) / len(lengths),
            color=COLORS[i], edgecolor="none", alpha=0.6, label=name)

ax.set_xscale("log")
ax.set_xlabel("Segment length (km)")
ax.set_ylabel("Fraction of segments")
ax.set_title(f"Segment length distribution — {' vs '.join(d[1] for d in datasets)} (normalized)")
ax.legend(loc="upper left", fontsize=8)

for i, (lengths, name) in enumerate(datasets):
    stats_text = (
        f"{name}\n"
        f"n = {len(lengths):,}\n"
        f"total  = {np.sum(lengths):,.0f} km\n"
        f"mean   = {np.mean(lengths):.3f} km\n"
        f"median = {np.median(lengths):.3f} km\n"
        f"P1  = {np.percentile(lengths, 1):.3f} km\n"
        f"P10 = {np.percentile(lengths, 10):.3f} km\n"
        f"P90 = {np.percentile(lengths, 90):.3f} km\n"
        f"P99 = {np.percentile(lengths, 99):.3f} km"
    )
    ax.text(0.97, 0.95 - i * 0.35, stats_text, transform=ax.transAxes,
            ha="right", va="top", fontsize=7, family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9))

fig.tight_layout()
fig.savefig("output_nhf_vs_links_lengths.png", dpi=200)
